# RAG with Hugging Face & FAISS

A simple Retrieval-Augmented Generation pipeline built with PDF extraction, text chunking, Sentence Transformers embeddings, FAISS vector search, and Hugging Face generation.

In [ ]:
!pip install -q faiss-cpu pymupdf sentence-transformers transformers torch

In [ ]:
import os
import fitz
import faiss
import numpy as np

from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [ ]:
pdf_path = input('Enter PDF path: ').strip()

def load_pdf(path):
    doc = fitz.open(path)
    text = ''
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

text = load_pdf(pdf_path)
print(len(text), 'characters extracted')

In [ ]:
def chunk_text(text, size=500):
    return [text[i:i+size] for i in range(0, len(text), size)]

chunks = chunk_text(text)
print('Chunks:', len(chunks))

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = embedding_model.encode(chunks)
embeddings = np.asarray(embeddings, dtype='float32')

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

print('FAISS vectors:', index.ntotal)

In [ ]:
query = input('Question: ')

query_embedding = embedding_model.encode([query])
query_embedding = np.asarray(query_embedding, dtype='float32')

_, ids = index.search(query_embedding, k=3)
retrieved = [chunks[i] for i in ids[0]]

context = '\n'.join(retrieved)
print(context[:1000])

In [ ]:
generator = pipeline('text-generation', model='gpt2')

prompt = f'Context:\n{context}\n\nQuestion: {query}\nAnswer:'

response = generator(prompt, max_new_tokens=100)
print(response[0]['generated_text'])

## Notes

This notebook demonstrates the core RAG workflow. It is an educational implementation and does not include production features such as reranking, metadata filtering, or citation tracking.